In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [2]:
import ase
from ase import Atoms
from ase.optimize import FIRE
from ase.io import read,write
from ase.constraints import FixAtoms

In [3]:
from copy import deepcopy

In [4]:
from matplotlib import cm

In [5]:
from nequip.ase import NequIPCalculator

In [6]:
sr_r45_layer2_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO_check/outputs/2025-09-30/15-13-51/best.ckpt'
lr_r45_layer2_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO_check/outputs/2025-09-30/14-50-34/best.ckpt'

sr_r55_layer2_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO_check/outputs/2025-10-06/16-10-50/best.ckpt'
lr_r55_layer2_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO_check/outputs/2025-09-30/15-53-37/best.ckpt'

sr_r45_layer3_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO/outputs/2025-09-02/16-40-07/best.ckpt'
lr_r45_layer3_lmax1 = '/global/scratch/users/dongjinkim/NequIP-LES/allegro-Au-MgO_check/outputs/2025-10-01/17-11-19/best.ckpt'


In [7]:
# 1: non-wetting
# 3: wetting


calculator = NequIPCalculator._from_checkpoint_model(ckpt_path=sr_r55_layer2_lmax1,
                                                    chemical_symbols = ['O', 'Mg', 'Al', 'Au'],
                                                    device='cuda' )

/global/home/users/dongjinkim/software/nequip/nequip/model/saved_models/checkpoint.py:68: UserWarning: `e3nn` versions differ between the checkpoint file (0.5.6) and the current run (0.5.7) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(


In [8]:
results = {}
results['Nequip'] = {}
results['DFT'] = {}

for configs in ['1-doped', '3-doped', '1-undoped', '3-undoped']:
    test_xyz = ase.io.read('./dft-optimized-struct/'+configs+'.xyz', '0')
    atomic_numbers = test_xyz.get_atomic_numbers()
    print(configs, "DFT energy:", test_xyz.info['energy'])
    results['DFT'][configs] = test_xyz.info['energy']

    substrate_index = [atom.index for atom in test_xyz if atom.symbol in ['Mg', 'O', 'Al']]
    au_index = [atom.index for atom in test_xyz if atom.symbol in ['Au']]

    atoms = deepcopy(test_xyz)

    # relax
    atoms.set_calculator(calculator)


    c = FixAtoms(indices=substrate_index)

    atoms.set_constraint(c)


    # Perform geometry optimization
    opt = FIRE(atoms, logfile=None)

    run = opt.run(fmax=0.01, steps=500)  # Adjust fmax yfor convergence criteria

    # write(dir_now+'relaxated-'+configs+'.xyz', atoms)

    print(configs, "Nequip energy:", atoms.get_potential_energy())
    results['Nequip'][configs] = atoms.get_potential_energy()
    print("--")


print(results)
# Nequip Doped $E_{wetting}-E_{nonwetting}$ (meV)
print("Nequip dopped dE [meV]:")
a = (results['Nequip']['3-doped'] - results['Nequip']['1-doped'])* 1000.
print(a)
# DFT Doped $E_{wetting}-E_{nonwetting}$ (meV)
print("Nequip undopped dE [meV]:")
a = (results['Nequip']['3-undoped'] - results['Nequip']['1-undoped'])* 1000.
print(a)


1-doped DFT energy: -1480190.9067329187
1-doped Nequip energy: -1480191.1358538007
--
3-doped DFT energy: -1480190.9736729292
3-doped Nequip energy: -1480190.7062563417
--
1-undoped DFT energy: -1476723.6744394903
1-undoped Nequip energy: -1476723.4516413622
--
3-undoped DFT energy: -1476722.7396195226
3-undoped Nequip energy: -1476723.022323225
--
{'Nequip': {'1-doped': -1480191.1358538007, '3-doped': -1480190.7062563417, '1-undoped': -1476723.4516413622, '3-undoped': -1476723.022323225}, 'DFT': {'1-doped': -1480190.9067329187, '3-doped': -1480190.9736729292, '1-undoped': -1476723.6744394903, '3-undoped': -1476722.7396195226}}
Nequip dopped dE [meV]:
429.59745903499424
Nequip undopped dE [meV]:
429.31813723407686


In [9]:
## DFT Doped $E_{wetting}-E_{nonwetting}$ (meV)
print("DFT dopped dE [meV]:")
(results['DFT']['3-doped'] - results['DFT']['1-doped'])* 1000.

DFT dopped dE [meV]:


-66.9400105252862

In [10]:
# DFT Doped $E_{wetting}-E_{nonwetting}$ (meV)
print("DFT undopped dE [meV]:")
(results['DFT']['3-undoped'] - results['DFT']['1-undoped'])* 1000.

DFT undopped dE [meV]:


934.8199677187949